# Download GCS keyframes to a reusable Kaggle Dataset

This notebook downloads AutoShot keyframes for batches `L21` through `L30` from Google Cloud Storage into `/kaggle/working`.

The notebook is intentionally data-only. It does not load an embedding model or extract vectors.

Output layout:

```text
/kaggle/working/
├── gcs_keyframes/
│   └── dataset=ai_challenge_2025/
│       └── batch=L21...L30/
│           └── profile=autoshot_v1/
│               └── video_id=<VIDEO_ID>/
│                   ├── shot_....jpg
│                   └── frames_manifest.jsonl
├── gcs_keyframes_index.jsonl
├── download_report.csv
├── download_summary.json
├── dataset_summary.json
└── README_DATASET.md
```

Before running:

1. Enable Internet for the Kaggle notebook.
2. Add the Kaggle Secret `GCS_CREDENTIALS_JSON`.
3. A GPU is not required.
4. Run `Save Version` → `Save & Run All` after reviewing the preflight size.


## 1. Install dependencies

In [ ]:
%pip install -q google-cloud-storage pandas tqdm

## 2. Imports

In [ ]:
from __future__ import annotations

import json
import logging
import os
import re
import shutil
import time

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path, PurePosixPath
from typing import Any, Iterable

import pandas as pd

from google.cloud import storage
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s - %(message)s",
)
logger = logging.getLogger("gcs_keyframe_download")


## 3. Configuration

`MAX_FRAMES_PER_BATCH = None` means every matching frame in each selected batch is downloaded.

The preflight checks the estimated saved output against `OUTPUT_BUDGET_GB`. If the data is larger than the budget, split the batch list across multiple notebook versions or datasets.


In [ ]:
GCS_BUCKET = "aic_ai_2026"
GCS_BUCKET_SECRET = "GCS_BUCKET"
GCS_CREDENTIALS_SECRET = "GCS_CREDENTIALS_JSON"

KEYFRAMES_PREFIX = "processed/keyframes"
DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"

SELECTED_BATCHES = [f"L{batch:02d}" for batch in range(21, 31)]
FRAME_TYPES = {"first", "middle", "last"}
MAX_FRAMES_PER_BATCH: int | None = None
DOWNLOAD_MANIFESTS = True

LOCAL_ROOT = Path("/kaggle/working/gcs_keyframes")
INDEX_PATH = Path("/kaggle/working/gcs_keyframes_index.jsonl")
REPORT_PATH = Path("/kaggle/working/download_report.csv")
ERRORS_PATH = Path("/kaggle/working/download_errors.jsonl")
SUMMARY_PATH = Path("/kaggle/working/download_summary.json")
DATASET_SUMMARY_PATH = Path("/kaggle/working/dataset_summary.json")
README_PATH = Path("/kaggle/working/README_DATASET.md")

DOWNLOAD_WORKERS = 16
DOWNLOAD_TIMEOUT_SEC = 300
SKIP_VALID_LOCAL_FILES = True
REQUIRE_ZERO_FAILURES = True

RUN_PREFLIGHT = True
OUTPUT_BUDGET_GB = 19.0
ENFORCE_OUTPUT_BUDGET = True

assert MAX_FRAMES_PER_BATCH is None or MAX_FRAMES_PER_BATCH > 0

logger.info("Selected batches: %s", SELECTED_BATCHES)
logger.info("Frame types: %s", sorted(FRAME_TYPES))
logger.info("Maximum frames per batch: %s", MAX_FRAMES_PER_BATCH)
logger.info("Download workers: %d", DOWNLOAD_WORKERS)
logger.info("Output root: %s", LOCAL_ROOT)


## 4. GCS authentication and path helpers

In [ ]:
FRAME_RE = re.compile(
    r"^shot_(?P<shot_index>\d{4})_"
    r"(?P<frame_type>first|middle|last)_"
    r"f(?P<frame_idx>\d{6})\.jpg$",
    flags=re.IGNORECASE,
)


def get_secret(name: str) -> str:
    '''Read a Kaggle Secret without printing its value.'''
    try:
        return UserSecretsClient().get_secret(name) or ""
    except Exception as exc:
        raise RuntimeError(f"Cannot read Kaggle Secret {name!r}: {exc}") from exc


def normalize_bucket_name(value: str) -> str:
    '''Accept a bare bucket name or a gs:// URI.'''
    value = value.strip().rstrip("/")
    return value[5:].split("/", 1)[0] if value.startswith("gs://") else value


def make_gcs_client() -> tuple[storage.Client, storage.Bucket, str]:
    '''Create an authenticated GCS client from Kaggle Secrets.'''
    credentials_json = get_secret(GCS_CREDENTIALS_SECRET)
    if not credentials_json:
        raise ValueError(f"Kaggle Secret {GCS_CREDENTIALS_SECRET!r} is empty")

    credentials_info = json.loads(credentials_json)
    credentials = service_account.Credentials.from_service_account_info(credentials_info)
    client = storage.Client(project=credentials.project_id, credentials=credentials)

    bucket_name = normalize_bucket_name(GCS_BUCKET or get_secret(GCS_BUCKET_SECRET))
    if not bucket_name:
        raise ValueError("GCS bucket name is empty")

    return client, client.bucket(bucket_name), bucket_name


def build_batch_prefix(batch_id: str) -> str:
    '''Build the exact keyframe prefix created by the frame-extraction notebook.'''
    return (
        f"{KEYFRAMES_PREFIX.strip('/')}/dataset={DATASET_ID}/"
        f"batch={batch_id.upper()}/profile={PROFILE_VERSION}/"
    )


def parse_partition_fields(object_name: str) -> dict[str, str]:
    '''Parse key=value partitions from a GCS object name.'''
    fields = {}
    for part in PurePosixPath(object_name).parts:
        if "=" in part:
            key, value = part.split("=", 1)
            fields[key] = value
    return fields


def local_path_for_object(object_name: str) -> tuple[Path, str]:
    '''Map a GCS object to its local and portable dataset paths.'''
    relative = PurePosixPath(object_name).relative_to(KEYFRAMES_PREFIX.strip("/"))
    local_path = LOCAL_ROOT.joinpath(*relative.parts)
    dataset_relative_path = (PurePosixPath("gcs_keyframes") / relative).as_posix()
    return local_path, dataset_relative_path


gcs_client, gcs_bucket, bucket_name = make_gcs_client()
logger.info("Authenticated bucket: %s", bucket_name)


## 5. Discover frames and manifests

JPEG objects are used as extraction inputs. Each video's `frames_manifest.jsonl` is also downloaded so the saved Kaggle Dataset remains self-contained and retains `fps`, `frame_sec`, shot boundaries, and GCS metadata.


In [ ]:
def frame_record_from_blob(blob: storage.Blob, bucket_name: str) -> dict[str, Any] | None:
    '''Convert one matching GCS JPEG object into a portable frame record.'''
    path = PurePosixPath(blob.name)
    match = FRAME_RE.fullmatch(path.name)
    if not match:
        return None

    frame_type = match.group("frame_type").lower()
    if frame_type not in FRAME_TYPES:
        return None

    fields = parse_partition_fields(blob.name)
    local_path, dataset_relative_path = local_path_for_object(blob.name)

    return {
        "object_type": "frame",
        "dataset_id": fields.get("dataset", ""),
        "batch_id": fields.get("batch", ""),
        "profile_version": fields.get("profile", ""),
        "video_id": fields.get("video_id", ""),
        "shot_index": int(match.group("shot_index")),
        "frame_type": frame_type,
        "frame_idx": int(match.group("frame_idx")),
        "gcs_uri": f"gs://{bucket_name}/{blob.name}",
        "object_name": blob.name,
        "size_bytes": int(blob.size or 0),
        "local_path": str(local_path),
        "relative_path": dataset_relative_path,
    }


def manifest_record_from_blob(blob: storage.Blob, bucket_name: str) -> dict[str, Any] | None:
    '''Convert a frames_manifest.jsonl object into a download record.'''
    if not blob.name.endswith("/frames_manifest.jsonl"):
        return None

    fields = parse_partition_fields(blob.name)
    local_path, dataset_relative_path = local_path_for_object(blob.name)

    return {
        "object_type": "manifest",
        "batch_id": fields.get("batch", ""),
        "video_id": fields.get("video_id", ""),
        "gcs_uri": f"gs://{bucket_name}/{blob.name}",
        "object_name": blob.name,
        "size_bytes": int(blob.size or 0),
        "local_path": str(local_path),
        "relative_path": dataset_relative_path,
    }


def list_batch_objects(
    client: storage.Client,
    bucket_name: str,
    batch_id: str,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    '''List selected frame and manifest objects for one batch.'''
    frames = []
    manifests = []

    for blob in client.list_blobs(bucket_name, prefix=build_batch_prefix(batch_id)):
        frame_record = frame_record_from_blob(blob, bucket_name)
        if frame_record is not None:
            frames.append(frame_record)
            continue

        if DOWNLOAD_MANIFESTS:
            manifest_record = manifest_record_from_blob(blob, bucket_name)
            if manifest_record is not None:
                manifests.append(manifest_record)

    if MAX_FRAMES_PER_BATCH is not None:
        frames = frames[:MAX_FRAMES_PER_BATCH]
        selected_videos = {record["video_id"] for record in frames}
        manifests = [record for record in manifests if record["video_id"] in selected_videos]

    return frames, manifests


## 6. Preflight size

This cell lists the selected GCS objects without downloading them. It reports per-batch frame counts and expected saved output size. The download is stopped early if the estimate exceeds the configured output budget.


In [ ]:
def run_preflight(
    client: storage.Client,
    bucket_name: str,
) -> pd.DataFrame:
    '''Estimate frame counts, manifest counts, and saved output size.'''
    rows = []

    for batch_id in tqdm(SELECTED_BATCHES, desc="Preflight", unit="batch"):
        frames, manifests = list_batch_objects(client, bucket_name, batch_id)
        frame_bytes = sum(record["size_bytes"] for record in frames)
        manifest_bytes = sum(record["size_bytes"] for record in manifests)

        rows.append(
            {
                "batch_id": batch_id,
                "frames": len(frames),
                "videos": len({record["video_id"] for record in frames}),
                "manifests": len(manifests),
                "frame_gb": frame_bytes / 1024**3,
                "total_gb": (frame_bytes + manifest_bytes) / 1024**3,
            }
        )

    report = pd.DataFrame(rows)
    total_gb = float(report["total_gb"].sum())
    free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3

    logger.info("Estimated saved output: %.2f GB", total_gb)
    logger.info("Current free working disk: %.2f GB", free_gb)
    display(report.round(3))

    if total_gb > free_gb * 0.95:
        raise RuntimeError(
            f"Estimated data size {total_gb:.2f} GB exceeds safe free disk {free_gb * 0.95:.2f} GB"
        )

    if ENFORCE_OUTPUT_BUDGET and total_gb > OUTPUT_BUDGET_GB:
        raise RuntimeError(
            f"Estimated output {total_gb:.2f} GB exceeds OUTPUT_BUDGET_GB={OUTPUT_BUDGET_GB:.2f}. "
            "Split SELECTED_BATCHES across multiple notebook versions or datasets."
        )

    return report


preflight_report = run_preflight(gcs_client, bucket_name) if RUN_PREFLIGHT else None


## 7. Parallel download pipeline

Downloads are atomic: data is written to a `.part` file and renamed only after completion. Existing files with the expected size are reused, so rerunning a failed notebook version resumes safely.


In [ ]:
def download_one(bucket: storage.Bucket, record: dict[str, Any]) -> str:
    '''Download one object atomically or reuse an already valid local file.'''
    destination = Path(record["local_path"])
    expected_size = int(record["size_bytes"])

    if SKIP_VALID_LOCAL_FILES and destination.exists():
        local_size = destination.stat().st_size
        if local_size > 0 and (expected_size <= 0 or local_size == expected_size):
            return "skipped"

    destination.parent.mkdir(parents=True, exist_ok=True)
    partial_path = destination.with_suffix(destination.suffix + ".part")

    try:
        bucket.blob(record["object_name"]).download_to_filename(
            str(partial_path),
            timeout=DOWNLOAD_TIMEOUT_SEC,
            checksum="auto",
        )

        if expected_size > 0 and partial_path.stat().st_size != expected_size:
            raise IOError(
                f"Size mismatch for {record['gcs_uri']}: expected={expected_size}, "
                f"downloaded={partial_path.stat().st_size}"
            )

        os.replace(partial_path, destination)
        return "downloaded"
    finally:
        partial_path.unlink(missing_ok=True)


def append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    '''Append dictionaries to a JSONL file.'''
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, allow_nan=True) + "\n")


def load_local_manifest_metadata(
    manifest_records: list[dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    '''Load all successfully downloaded manifest rows for one batch.'''
    metadata = {}

    for record in manifest_records:
        manifest_path = Path(record["local_path"])
        if not manifest_path.exists():
            continue

        with manifest_path.open("r", encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue
                row = json.loads(line)
                object_name = str(row.get("image_storage_key") or "")
                if object_name:
                    metadata[object_name] = row

    return metadata


def enrich_frame_records(
    frame_records: list[dict[str, Any]],
    manifest_records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    '''Join portable frame records with locally downloaded manifest metadata.'''
    metadata = load_local_manifest_metadata(manifest_records)
    enriched = []

    for record in sorted(frame_records, key=lambda item: item["object_name"]):
        row = metadata.get(record["object_name"], {})
        item = {key: value for key, value in record.items() if key != "local_path"}
        item.update(
            {
                "frame_sec": row.get("frame_sec", ""),
                "fps": row.get("fps", ""),
                "shot_id": row.get("shot_id", ""),
                "shot_start_frame": row.get("shot_start_frame", ""),
                "shot_end_frame": row.get("shot_end_frame", ""),
                "keyframe_id": row.get("keyframe_id", ""),
            }
        )
        enriched.append(item)

    return enriched


def download_batch(
    client: storage.Client,
    bucket: storage.Bucket,
    bucket_name: str,
    batch_id: str,
) -> dict[str, Any]:
    '''Download all selected frames and manifests for one batch.'''
    frames, manifests = list_batch_objects(client, bucket_name, batch_id)
    objects = frames + manifests

    counts = {"downloaded": 0, "skipped": 0, "failed": 0}
    ready_frames = []
    ready_manifests = []
    downloaded_bytes = 0
    errors = []
    started = time.perf_counter()

    logger.info(
        "%s selected frames=%d manifests=%d objects=%d",
        batch_id,
        len(frames),
        len(manifests),
        len(objects),
    )

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        futures = {executor.submit(download_one, bucket, record): record for record in objects}

        with tqdm(total=len(objects), desc=f"Download {batch_id}", unit="object") as progress:
            for future in as_completed(futures):
                record = futures[future]
                try:
                    status = future.result()
                    counts[status] += 1
                    if status == "downloaded":
                        downloaded_bytes += int(record["size_bytes"])

                    if record["object_type"] == "frame":
                        ready_frames.append(record)
                    else:
                        ready_manifests.append(record)
                except Exception as exc:
                    counts["failed"] += 1
                    errors.append(
                        {
                            "batch_id": batch_id,
                            "object_name": record["object_name"],
                            "gcs_uri": record["gcs_uri"],
                            "error": repr(exc),
                        }
                    )
                progress.update(1)

    append_jsonl(INDEX_PATH, enrich_frame_records(ready_frames, ready_manifests))
    append_jsonl(ERRORS_PATH, errors)

    elapsed = time.perf_counter() - started
    network_mb_s = downloaded_bytes / 1024**2 / elapsed if downloaded_bytes and elapsed else 0.0

    return {
        "batch_id": batch_id,
        "selected_frames": len(frames),
        "selected_manifests": len(manifests),
        "available_frames": len(ready_frames),
        "available_manifests": len(ready_manifests),
        **counts,
        "downloaded_gb": downloaded_bytes / 1024**3,
        "elapsed_sec": elapsed,
        "network_mb_s": network_mb_s,
    }


def run_download() -> tuple[pd.DataFrame, dict[str, Any]]:
    '''Download L21-L30 sequentially and persist progress after each batch.'''
    client, bucket, bucket_name = make_gcs_client()
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    for path in (INDEX_PATH, REPORT_PATH, ERRORS_PATH, SUMMARY_PATH):
        path.unlink(missing_ok=True)

    reports = []
    started = time.perf_counter()

    for batch_id in SELECTED_BATCHES:
        report = download_batch(client, bucket, bucket_name, batch_id)
        reports.append(report)
        pd.DataFrame(reports).to_csv(REPORT_PATH, index=False)
        logger.info(
            "%s complete frames=%d failed=%d network=%.2f MB/s",
            batch_id,
            report["available_frames"],
            report["failed"],
            report["network_mb_s"],
        )

    report_table = pd.DataFrame(reports)
    failed_objects = int(report_table["failed"].sum())
    elapsed = time.perf_counter() - started

    summary = {
        "status": "success" if failed_objects == 0 else "completed_with_errors",
        "bucket": bucket_name,
        "batches": SELECTED_BATCHES,
        "frames": int(report_table["available_frames"].sum()),
        "manifests": int(report_table["available_manifests"].sum()),
        "failed_objects": failed_objects,
        "elapsed_sec": round(elapsed, 2),
        "output_root": str(LOCAL_ROOT),
        "index_path": str(INDEX_PATH),
    }
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    if REQUIRE_ZERO_FAILURES and failed_objects:
        raise RuntimeError(
            f"Download completed with {failed_objects} failed objects. Review {ERRORS_PATH} and rerun."
        )

    return report_table, summary


## 8. Run the full L21-L30 download

In [ ]:
download_report, download_summary = run_download()

display(download_report.round(3))
print(json.dumps(download_summary, indent=2))


## 9. Validate and describe the reusable dataset

The final index stores `relative_path`, not the temporary `/kaggle/working` path. This makes it portable after the output is converted to a Kaggle Dataset and mounted under `/kaggle/input/<dataset-slug>`.


In [ ]:
def directory_size_bytes(path: Path) -> int:
    '''Calculate the total size of all files below a directory.'''
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())


def validate_dataset() -> tuple[pd.DataFrame, dict[str, Any]]:
    '''Validate local objects and write portable dataset documentation.'''
    if not INDEX_PATH.exists():
        raise FileNotFoundError(f"Missing dataset index: {INDEX_PATH}")

    records = []
    with INDEX_PATH.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    index_table = pd.DataFrame(records)
    if index_table.empty:
        raise RuntimeError("Dataset index is empty")

    local_frames = list(LOCAL_ROOT.rglob("*.jpg"))
    local_manifests = list(LOCAL_ROOT.rglob("frames_manifest.jsonl"))
    missing_index_files = 0

    for relative_path in index_table["relative_path"]:
        relative = PurePosixPath(relative_path)
        local_path = Path("/kaggle/working").joinpath(*relative.parts)
        missing_index_files += int(not local_path.is_file())

    batch_validation = (
        index_table.groupby("batch_id")
        .agg(frames=("object_name", "count"), videos=("video_id", "nunique"))
        .reset_index()
    )

    summary = {
        "dataset_id": DATASET_ID,
        "profile_version": PROFILE_VERSION,
        "batches": SELECTED_BATCHES,
        "frames": len(local_frames),
        "indexed_frames": len(index_table),
        "videos": int(index_table["video_id"].nunique()),
        "manifests": len(local_manifests),
        "missing_index_files": missing_index_files,
        "size_gb": directory_size_bytes(LOCAL_ROOT) / 1024**3,
        "frame_root": "gcs_keyframes",
        "index_file": INDEX_PATH.name,
    }

    if len(local_frames) != len(index_table) or missing_index_files:
        raise RuntimeError(
            f"Validation failed: local_frames={len(local_frames)}, indexed_frames={len(index_table)}, "
            f"missing_index_files={missing_index_files}"
        )

    DATASET_SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    readme = f'''# AI Challenge 2025 AutoShot keyframes L21-L30

Frames and per-video manifests downloaded from GCS for reusable Kaggle feature extraction.

## Contents

- Frames: {summary['frames']:,}
- Videos: {summary['videos']:,}
- Manifests: {summary['manifests']:,}
- Size: {summary['size_gb']:.2f} GB
- Profile: {PROFILE_VERSION}

## Load from a future Kaggle notebook

```python
import json
from pathlib import Path, PurePosixPath

DATASET_ROOT = Path('/kaggle/input/<your-dataset-slug>')
INDEX_PATH = DATASET_ROOT / 'gcs_keyframes_index.jsonl'

records = []
with INDEX_PATH.open('r', encoding='utf-8') as file:
    for line in file:
        record = json.loads(line)
        relative = PurePosixPath(record['relative_path'])
        record['local_path'] = str(DATASET_ROOT.joinpath(*relative.parts))
        records.append(record)
```
'''
    README_PATH.write_text(readme, encoding="utf-8")

    return batch_validation, summary


batch_validation, dataset_summary = validate_dataset()

display(batch_validation)
print(json.dumps(dataset_summary, indent=2))


## 10. Save the data as a Kaggle Dataset

After all validation cells pass:

1. Click **Save Version**.
2. Select **Save & Run All**.
3. Wait for the committed notebook version to finish successfully.
4. Open that version's **Output**.
5. Choose **Create Dataset** or select this notebook output from the Kaggle Dataset uploader.

Do not create a ZIP inside `/kaggle/working`; JPEG files are already compressed and the ZIP would duplicate the saved output size.
